# nb_01_metadata_delta — scan S3 shortcut, detect changes, maintain `file_metadata`

**Pipeline 1.** Recursively lists the S3 shortcut (depth-agnostic, Spark-native — no driver-side
walk), computes a change hash, and MERGEs into `file_metadata`, driving the `process_status`
state machine. Also detects deletions via a stale `last_seen_utc`.

See `PRODUCT_SPEC.md` sections 5, 7.1, 9, 12.


## Parameters + config


In [ ]:
from datetime import datetime, timezone

cfg = {r['key']: r['value'] for r in spark.table('config').collect()}
SHORTCUT_ROOT = cfg.get('shortcut_root', 'Files/s3_mmx_bucket')
RUN_START = datetime.now(timezone.utc)
print('scanning:', SHORTCUT_ROOT, '| run_start:', RUN_START)


## Recursive, Spark-native listing
`binaryFile` with `recursiveFileLookup` distributes the walk across the cluster and handles
deeply nested folders. We select only `path`/`length`/`modificationTime` so file **bytes are
never read** (column pruning) — cheap even for very large corpora.


In [ ]:
from pyspark.sql import functions as F

raw = (spark.read.format('binaryFile')
       .option('recursiveFileLookup', 'true')
       .load(SHORTCUT_ROOT)
       .select('path', 'length', 'modificationTime'))

scanned = (raw
    .withColumn('file_path', F.col('path'))
    .withColumn('file_name', F.element_at(F.split('path', '/'), -1))
    .withColumn('file_extension',
                F.lower(F.regexp_extract(F.col('file_name'), r'\.([^.]+)$', 1)))
    .withColumn('file_size', F.col('length').cast('bigint'))
    .withColumn('modified_datetime', F.col('modificationTime'))
    # etag/content_hash are not available from the listing; size+mtime is the change signal.
    .withColumn('etag', F.lit(None).cast('string'))
    .withColumn('author', F.lit(None).cast('string'))
    .withColumn('content_hash', F.lit(None).cast('string'))
    .withColumn('change_hash',
                F.sha2(F.concat_ws('|', F.col('file_path'), F.col('file_size'),
                                   F.col('modified_datetime').cast('string')), 256))
    .withColumn('last_seen_utc', F.lit(RUN_START))
    .select('file_path','file_name','file_extension','file_size','modified_datetime',
            'author','etag','content_hash','change_hash','last_seen_utc'))

# Cache: this DataFrame feeds several downstream actions (count, the change MERGE, and the
# deletion sweep). Caching avoids recomputing the listing + transforms each time.
scanned = scanned.cache()

print('files discovered:', scanned.count())
scanned.select('file_path','file_size','change_hash').show(5, truncate=False)


## MERGE into `file_metadata` (state machine)
- new path → `new`
- existing path, `change_hash` changed → `reingest`
- existing path, unchanged → keep status, just refresh `last_seen_utc`


In [ ]:
from delta.tables import DeltaTable

now_str = RUN_START
tgt = DeltaTable.forName(spark, 'file_metadata')

(tgt.alias('t')
  .merge(scanned.alias('s'), 't.file_path = s.file_path')
  # changed content -> reingest
  .whenMatchedUpdate(
      condition='t.change_hash <> s.change_hash',
      set={
        'file_size': 's.file_size', 'modified_datetime': 's.modified_datetime',
        'file_name': 's.file_name', 'file_extension': 's.file_extension',
        'etag': 's.etag', 'change_hash': 's.change_hash',
        'last_seen_utc': 's.last_seen_utc',
        'process_status': F.lit('reingest'),
        'status_reason': F.lit(None).cast('string'),
        'status_updated_utc': F.lit(now_str),
      })
  # unchanged -> just mark as seen this run
  .whenMatchedUpdate(
      condition='t.change_hash = s.change_hash',
      set={'last_seen_utc': 's.last_seen_utc'})
  # brand new file
  .whenNotMatchedInsert(
      values={
        'file_path': 's.file_path', 'file_name': 's.file_name',
        'file_extension': 's.file_extension', 'file_size': 's.file_size',
        'modified_datetime': 's.modified_datetime', 'author': 's.author',
        'etag': 's.etag', 'content_hash': 's.content_hash',
        'change_hash': 's.change_hash', 'acl_version': F.lit(None).cast('string'),
        'last_seen_utc': 's.last_seen_utc',
        'process_status': F.lit('new'),
        'status_reason': F.lit(None).cast('string'),
        'retry_count': F.lit(0),
        'status_updated_utc': F.lit(now_str),
      })
  .execute())
print('merge complete')


## Deletion detection
Any row not seen in this scan (stale `last_seen_utc`) that isn't already `deleted` is flagged
`deleted`; nb_03 will purge its chunks from AI Search.


In [ ]:
(DeltaTable.forName(spark, 'file_metadata').alias('t')
  .merge(scanned.select('file_path').alias('s'), 't.file_path = s.file_path')
  .whenNotMatchedBySourceUpdate(
      condition="t.process_status <> 'deleted'",
      set={'process_status': F.lit('deleted'),
           'status_reason': F.lit('not_present_in_source'),
           'status_updated_utc': F.lit(now_str)})
  .execute())
print('deletion sweep complete')


## Summary


In [ ]:
(spark.table('file_metadata')
   .groupBy('process_status').count().orderBy('process_status').show())
